# Kadane (Java)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/Kadane_java.ipynb)

Capitolo 18 — Algoritmi ricorsivi e iterativi a confronto. Notebook eseguibile di `Kadane`: l'algoritmo in Java, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/18-ricorsivo-iterativo/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/Kadane_python.ipynb) e [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/Kadane_cpp.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo_java.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. La prima cella verifica che il JDK ci sia e, se manca, lo installa. Ogni programma Java è scritto su un file dalla cella `%%writefile`, poi compilato con `javac` ed eseguito con `java` nella cella successiva.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `FibRec` → `fibRec` (camelCase).

## Preparazione

Su Colab il JDK di solito c'è già: la cella qui sotto lo verifica e, se manca, lo installa.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Kadane: un algoritmo iterativo migliore

**Kadane**$(a,n)$ — *Input*: un array $a[1..n]$ di $n\ge1$ interi. *Output*: la somma massima di un sottoarray non vuoto $a[l..r]$. È la versione compatta del capitolo: $cur\leftarrow\max(a[i],\,cur+a[i])$, $best\leftarrow\max(best,cur)$. Il ciclo fa $n-1$ iterazioni: $\Theta(n)$ passi e $\Theta(1)$ memoria, contro le $n(n+1)/2$ coppie $(l,r)$ della forza bruta ($\Theta(n^2)$) e contro il divide et impera ricorsivo ($\Theta(n\log_2 n)$): un buon algoritmo iterativo può battere la ricorsione.

Sull'esempio delle variazioni di prezzo del capitolo 17 il risultato è $43$. Poi, su array pseudo-casuali con valori fra $-50$ e $50$ (stesso generatore in tutti i linguaggi, seme $n$), le iterazioni di `Kadane`, le coppie esaminate dalla forza bruta e la verifica che i due algoritmi trovino la stessa somma massima; infine i tempi misurati.

In [2]:
%%writefile KadaneCompact.java
import java.util.*;

public class KadaneCompact {
    // kadane(a, n): somma massima di un sottoarray non vuoto di a[0..n-1]
    static int kadane(int[] a, int n) {
        int best = a[0];
        int cur = a[0];                 // somma massima di un sottoarray che termina in i
        int i = 1;
        while (i < n) {
            cur = Math.max(a[i], cur + a[i]);
            best = Math.max(best, cur);
            i = i + 1;
        }
        return best;
    }

    // Forza bruta: restituisce {somma massima, coppie (l, r) esaminate}
    static long[] bruteForceBest(int[] a, int n) {
        int best = a[0];
        long pairs = 0;
        for (int l = 0; l < n; l++) {
            int s = 0;
            for (int r = l; r < n; r++) {
                s = s + a[r];
                pairs++;
                if (s > best) best = s;
            }
        }
        return new long[]{best, pairs};
    }

    // Interi fra -50 e 50 (generatore congruenziale lineare)
    static int[] randomValues(int n, long seed) {
        long x = seed;
        int[] a = new int[n];
        for (int i = 0; i < n; i++) {
            x = (x * 1103515245L + 12345L) % 2147483648L;
            a[i] = (int) (x % 101L) - 50;
        }
        return a;
    }

    static double nowMs() {
        return System.nanoTime() / 1e6;
    }

    public static void main(String[] args) {
        int[] a = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
        System.out.println("a = " + Arrays.toString(a));
        System.out.println("Kadane(a, 16) = " + kadane(a, 16));
        int[] b = {-3, -1, -2, -5};
        System.out.println("a = " + Arrays.toString(b) + "  Kadane(a, 4) = " + kadane(b, 4));
        System.out.println();
        System.out.printf("%6s %7s %11s %11s %9s  stessa somma?%n", "n", "Kadane", "BruteForce", "iterations", "pairs");
        double[][] times = new double[3][];
        int idx = 0;
        for (int n : new int[]{1000, 2000, 4000}) {
            int[] v = randomValues(n, n);
            double t0 = nowMs(); int k = kadane(v, n); double t1 = nowMs() - t0;
            t0 = nowMs(); long[] bf = bruteForceBest(v, n); double t2 = nowMs() - t0;
            System.out.printf("%6d %7d %11d %11d %9d  %s%n", n, k, bf[0], n - 1, bf[1], k == bf[0] ? "OK" : "ERROR");
            times[idx++] = new double[]{n, t1, t2};
        }
        System.out.println();
        System.out.println("Tempi misurati (dipendono dalla macchina):");
        for (double[] t : times) {
            System.out.printf("n=%-6d Kadane %9.4f ms   BruteForce %10.3f ms%n", (int) t[0], t[1], t[2]);
        }
    }
}

Writing KadaneCompact.java


In [3]:
!javac KadaneCompact.java && java KadaneCompact

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
Kadane(a, 16) = 43
a = [-3, -1, -2, -5]  Kadane(a, 4) = -1

     n  Kadane  BruteForce  iterations     pairs  stessa somma?
  1000     547         547         999    500500  OK
  2000     981         981        1999   2001000  OK
  4000    1955        1955        3999   8002000  OK

Tempi misurati (dipendono dalla macchina):
n=1000   Kadane    0.2381 ms   BruteForce      6.698 ms
n=2000   Kadane    0.4051 ms   BruteForce     10.358 ms
n=4000   Kadane    0.2869 ms   BruteForce      3.305 ms


**Prova tu.** Cambia $n$: le coppie della forza bruta crescono come $n^2$, le iterazioni di Kadane come $n$.

In [4]:
%%writefile ProvaKadaneCompact.java
import java.util.*;

public class ProvaKadaneCompact {
    public static void main(String[] args) {
        // Prova con n = 8000: le coppie della forza bruta sono n(n+1)/2 = 32 004 000, le iterazioni di Kadane 7999.
        int n = 8000;
        int[] v = KadaneCompact.randomValues(n, 12345);
        long[] bf = KadaneCompact.bruteForceBest(v, n);
        System.out.println(KadaneCompact.kadane(v, n) + " (" + bf[0] + ", " + bf[1] + ")");
    }
}

Writing ProvaKadaneCompact.java


In [5]:
!javac ProvaKadaneCompact.java && java ProvaKadaneCompact

1903 (1903, 32004000)


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).